## ¿Qué es Stacking (Apilamiento)?

El **Stacking (Apilamiento)** es una técnica avanzada de _ensemble learning_ (aprendizaje de conjuntos) que busca mejorar la precisión predictiva combinando las fortalezas de múltiples modelos de machine learning. A diferencia de otras técnicas como Bagging o Boosting, que a menudo usan modelos del mismo tipo, Stacking puede integrar una variedad de modelos heterogéneos.

### Idea Principal:

La idea central del Stacking es entrenar un modelo final, conocido como **meta-modelo** o **estimador final**, para aprender a hacer una predicción final combinando las predicciones generadas por varios **modelos base**. Esencialmente, las predicciones de los modelos base se utilizan como nuevas características de entrada para el meta-modelo.

### Componentes Principales:

1.  **Modelos Base (o Nivel 0)**:
    *   Son los modelos iniciales que se entrenan directamente sobre los datos de entrenamiento originales. Pueden ser de cualquier tipo (e.g., regresión lineal, árboles de decisión, máquinas de vectores de soporte, redes neuronales, etc.).
    *   Cada modelo base aprende a resolver el problema de predicción de una manera ligeramente diferente, capturando distintas relaciones en los datos.
    *   Sus predicciones se convierten en las "características" o "entradas" para el siguiente nivel.

2.  **Meta-Modelo (o Nivel 1)**:
    *   Es el modelo que aprende a "decidir" cómo combinar las predicciones de los modelos base.
    *   Se entrena sobre un nuevo conjunto de datos donde las entradas son las predicciones de los modelos base (generadas a menudo mediante validación cruzada para evitar el _data leakage_) y las salidas son las etiquetas verdaderas del conjunto de entrenamiento.
    *   El meta-modelo tiene la tarea de ponderar o transformar las predicciones de los modelos base para producir la predicción final, aprendiendo qué modelo base es más confiable en diferentes situaciones.

### Python:
```python
  from sklearn.ensemble import RandomForestClassifier, StackingClassifier
  from sklearn.linear_model import LogisticRegression
  from sklearn.svm import SVC

  estimators = [
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
    ('svc', SVC(probability=True, random_state=42))
  ]

  # Meta-modelo
  final_estimator = LogisticRegression()

  # Stacking
  stack = StackingClassifier(
    estimators=estimators,
    final_estimator=final_estimator,
    cv=5,  # Muy importante
    passthrough=True # le pasas también las variables orginales
  )
```


In [4]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, matthews_corrcoef, classification_report, confusion_matrix

In [6]:
wine = load_wine()
X = wine.data
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 1. Initialize base models
svc = SVC(random_state=42, probability=True)
rf_clf = RandomForestClassifier(random_state=42)
knn_clf = KNeighborsClassifier(n_neighbors=5)

# 2. Create a list of base models (estimators)
estimators = [
    ('svc', svc),
    ('rf', rf_clf),
    ('knn', knn_clf)
]

# 3. Initialize the meta-classifier
meta_classifier = LogisticRegression(solver='liblinear', random_state=42)

# 4. Initialize the StackingClassifier
stk_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_classifier,
    cv=5, # Use 5-fold cross-validation to generate meta-features
    passthrough=True # Pass original features to the final estimator as well
)

# 5. Train the StackingClassifier
stk_clf.fit(X_train, y_train)

print("StackingClassifier trained successfully.")



y_pred_stk = stk_clf.predict(X_test)

# 2. Importa la función accuracy_score del módulo sklearn.metrics. (Already imported in this block)

# 3. Calcula la precisión del modelo de stacking comparando y_test con y_pred_stk usando accuracy_score.
mcc = matthews_corrcoef(y_test, y_pred_stk)

# 4. Imprime la precisión obtenida, formateando la salida para que sea clara.
print(f"MCC del StackingClassifier en el conjunto de prueba: {mcc:.4f}")

print("\nReporte de Clasificación:\n")
print(classification_report(y_test, y_pred_stk, target_names=wine.target_names))

print("\nMatriz de Confusión:\n")
print(confusion_matrix(y_test, y_pred_stk))


StackingClassifier trained successfully.
MCC del StackingClassifier en el conjunto de prueba: 1.0000

Reporte de Clasificación:

              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        19
     class_1       1.00      1.00      1.00        21
     class_2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54


Matriz de Confusión:

[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]
